In [ ]:
# =============================================================================
# Fig4 (v2): Patient counts (Top) + Bed OCCUPANCY RATE (Middle) + Income bars (Bottom)
#
# KEY CHANGE from the original script:
#   Row 2 no longer plots a raw patient-to-bed RATIO (mo_total/beds). Instead
#   it plots BED OCCUPANCY RATE, computed disease-by-disease as:
#
#       Occupancy = sum_over_diseases( patients_disease * LOS_disease ) / (beds * 365)
#
#   where LOS_disease is the average length of stay (days) for that disease
#   (Table S3), and patients_disease is either:
#       - local_patient_d(city)                      -> "before mobility"
#       - local_patient_d(city) + net_inflow_d(city)  -> "after mobility"
#   Row 1 (raw patient counts, before/after/net) is UNCHANGED and still uses
#   the disease-aggregated totals, since the user only asked to change the
#   ratio panel.
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as mcm
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.gridspec import GridSpec
from shapely.geometry import box as shapely_box
from pyproj import Transformer

# ── 0. Paths & settings ───────────────────────────────────────────────────────
FLOW_DIR   = Path("/Volumes/UCL/论文工作/空气污染/cross_flow_truncated/averaged_results/flow_avg")
LOCAL_DIR  = Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/city_patient_sum")

# NEW: per-disease sources — BOTH now come from Step 1 + Step 2 outputs directly
# (Step 1's citysum_bydisease file already contains local_mo_total per city/disease,
# so the separate Step 1b "localsum_bydisease" step is no longer needed.)
CITYSUM_BYDISEASE_DIR = Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/citysum_bydisease")
CITYSUM_BYDISEASE_TPL = "citysum_bydisease_{air}_{year}.csv"

DISEASE_FLOW_DIR = Path("/Volumes/UCL/论文工作/空气污染/cross_flow_by_disease/flow_results")
DISEASE_FLOW_TPL = "flow_patientnum_{disease_tag}_{air}_{year}.csv"

INPUT_FILE = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/air_pollution/data source/hospital/13-National hospital directory.xlsx")
SHP_PATH   = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/city_shp/shi_en.shp")
CHINA_SHP  = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国底图-中图社审过版本/中国底图/中国面.shp")
CHINA_SHP2 = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国国界线/九段线/九段线和群岛.shp")
OUTFILE    = Path("/Users/shirley/Desktop/plots_V2/Fig4_occupancy.png")

INCOME_DIR       = Path("/Volumes/UCL/论文工作/空气污染/weighted_gdp/avg_fer_rcp/avg_2020.csv")
INCOME_CITY_COL  = "city"
INCOME_CLASS_COL = "income_group"
GDP_CLASS_ORDER  = ["Low", "Lower middle", "Middle", "Upper middle", "High"]
GDP_CLASS_SHORT  = {"Low": "Low", "Lower middle": "L-Mid", "Middle": "Mid",
                     "Upper middle": "U-Mid", "High": "High"}
GDP_CLASS_COLORS = ["#D6EFD6", "#A8DDA8", "#6FC26F", "#3B9E3B", "#1B6B1B"]

SCENARIO = "earlypeak_NZ_CL"
YEARS    = [2020, 2030, 2040, 2050, 2060]

# ── 0b. Length-of-stay lookup (Table S3) ──────────────────────────────────────
# Average length of stay per hospital admission, in days, for each of the
# five PM2.5-attributable diseases.
LOS_BY_SHORT_NAME = {
    "COPD":   10.0,
    "IHD":    12.6,
    "LRI":    11.0,
    "LC":     14.2,
    "STROKE": 19.7,
}

def match_los(disease_name: str) -> float:
    """
    Robustly map a full disease-name string (as it appears in your
    citysum_bydisease / localsum_bydisease files) to its length-of-stay
    value, using keyword matching so it's insensitive to exact wording
    (e.g. 'Ischemic Heart Disease' vs 'Ischaemic Heart Disease' vs 'IHD').

    >>> EDIT/EXTEND the keyword lists below if your disease-name strings
        don't match any of these patterns — the function will raise a
        clear error rather than silently returning a wrong value.
    """
    s = disease_name.strip().lower()
    if "copd" in s or "chronic obstructive" in s:
        return LOS_BY_SHORT_NAME["COPD"]
    if "isch" in s or "ihd" in s or "coronary" in s or "chd" in s:
        return LOS_BY_SHORT_NAME["IHD"]
    if "lower respiratory" in s or s == "lri" or "lri" in s:
        return LOS_BY_SHORT_NAME["LRI"]
    if "lung cancer" in s or s == "lc":
        return LOS_BY_SHORT_NAME["LC"]
    if "stroke" in s:
        return LOS_BY_SHORT_NAME["STROKE"]
    raise ValueError(
        f"Could not match disease name '{disease_name}' to a known LOS category. "
        f"Please extend match_los() with an appropriate keyword."
    )

CITY_NAME_MAP = {
    "Wulumuqi":   "Urumqi",
    "Xian":       "Xi'an",
    "Qiqihaer":   "Qiqihar",
    "Huhehaote":  "Hohhot",
    "Haerbin":    "Harbin",
    "Xiuqian":    "Suqian",
    "Wulanchabu": "Ulanqab",
    "Shaoang":    "Zhaotong",
    "Tongcuan":   "Tongchuan",
    "Xiangfan":   "Xiangyang",
    "Akesu":        "Aksu",
    "Alaer":        "Alar",
    "Alashan":      "Alxa",
    "Bayanzhuoer":  "Bayannur",
    "Bayinguoleng": "Bayingolin",
    "Boertalameng": "Bortala",
    "Changdu":      "Qamdo",
    "Eerduosi":     "Erdos",
    "Ordos":      "Erdos",
    "Guoluo":       "Golog",
    "Guolo":      "Golog",
    "Hulunbeier":   "Hulunbuir",
    "Kezilesu":     "Kizilsu",
    "Ledong":       "Ledong",
    "Lingshui":     "Lingshui",
    "Linzhi":       "Nyingchi",
    "Naqu":         "Nagqu",
    "Qiongzhong":   "Qiongzhong",
    "Shennongjia":  "Shennongjia",
    "Tumushuke":    "Tumxuk",
    "Xilinguole":   "Xilingol",
    "Xingan":       "Hinggan",
}

C_EAST   = "#C0392B"
C_WEST   = "#2471A3"
C_NODATA = "#DDDDDD"
PROJ_STR = "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105"

# ── 1. Global style ───────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":      "Arial",
    "font.size":        6,
    "axes.titlesize":   6,
    "axes.titleweight": "bold",
    "axes.titlepad":    2,
})

# ── 2. Spatial data ───────────────────────────────────────────────────────────
china_border = gpd.read_file(CHINA_SHP).to_crs(PROJ_STR)
jiudanline   = gpd.read_file(CHINA_SHP2).to_crs(PROJ_STR)

city_shp_raw = gpd.read_file(SHP_PATH)
city_shp_raw["English"] = city_shp_raw["English"].str.strip().map(
    lambda x: CITY_NAME_MAP.get(x, x))
city_shp = city_shp_raw.to_crs(PROJ_STR)

_hhy_transformer = Transformer.from_crs("EPSG:4326", PROJ_STR, always_xy=True)
_HHY_X, _HHY_Y  = _hhy_transformer.transform([127.5, 98.5], [50.2, 25.0])

_NANHAI_BOUNDS = (
    gpd.GeoDataFrame(geometry=[shapely_box(105, 2, 122, 24)], crs="EPSG:4326")
    .to_crs(PROJ_STR).total_bounds
)

_hhy_x0, _hhy_y0 = _HHY_X[0], _HHY_Y[0]
_hhy_x1, _hhy_y1 = _HHY_X[1], _HHY_Y[1]

def _hhy_x_at_y(y):
    t = (y - _hhy_y1) / (_hhy_y0 - _hhy_y1)
    return _hhy_x1 + t * (_hhy_x0 - _hhy_x1)

_cx = city_shp.geometry.centroid.x
_cy = city_shp.geometry.centroid.y
city_shp["region"] = np.where(_cx > _hhy_x_at_y(_cy), "East", "West")

region_map = (
    city_shp[["English", "region"]]
    .drop_duplicates(subset="English")
    .set_index("English")["region"]
    .to_dict()
)

# ── 3. Data loaders (row-1 aggregate patient counts — unchanged) ─────────────
def rename_idx(idx):
    return idx.str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))

def load_flow_matrix(year):
    path = FLOW_DIR / f"flow_patientnum_avg_{SCENARIO}_{year}.csv"
    df   = pd.read_csv(path, index_col=0)
    df.index   = rename_idx(df.index)
    df.columns = rename_idx(df.columns)
    df = df.loc[~df.index.isin(["total"]), ~df.columns.isin(["total"])]
    np.fill_diagonal(df.values, 0)
    return df

def load_citysum(year):
    path = LOCAL_DIR / f"citysum_{SCENARIO}_{year}.csv"
    df   = pd.read_csv(path)
    df["city"] = df["city"].str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    return df.groupby("city")[["local_patient", "mo_total"]].sum()

def compute_year(year):
    """
    Row-1 aggregate patient counts.

    FIX (previously double-subtracted outflow): `local_patient` in the old
    citysum file is ALREADY net of outflow (verified against the new
    per-disease pipeline: local_old == local_new == mo_total - outflow).
    So "after mobility" demand should be:
        demand = local_patient + inflow
    NOT:
        demand = local_patient + net   (= local_patient + inflow - outflow)
    which double-subtracted outflow and systematically understated demand
    in high-throughput hub cities (large in AND out flows).
    """
    df_flow  = load_flow_matrix(year)
    df_local = load_citysum(year)

    inflow   = df_flow.sum(axis=0).groupby(level=0).sum()
    outflow  = df_flow.sum(axis=1).groupby(level=0).sum()

    all_cities = inflow.index.union(outflow.index)
    inflow   = inflow.reindex(all_cities,  fill_value=0)
    outflow  = outflow.reindex(all_cities, fill_value=0)

    net = (inflow - outflow).rename("net")   # kept for reference/diagnostics only

    df_local = df_local.groupby(level=0).sum()
    common   = net.index.intersection(df_local.index)
    out = df_local.loc[common].copy()

    out["net"]    = net.loc[common].values
    out["inflow"] = inflow.reindex(out.index, fill_value=0).values

    raw_demand = out["local_patient"] + out["inflow"]
    out["demand"] = np.where(raw_demand < 0, 0.0, raw_demand)

    if "mo_total" in out.columns:
        out["mo_total"] = np.where(out["mo_total"] < 0, 0.0, out["mo_total"])

    return out

# ── 3b. NEW: per-disease patient-day loaders (drives Row 2 occupancy) ────────
def load_disease_flow_matrix(year, disease_tag):
    path = DISEASE_FLOW_DIR / DISEASE_FLOW_TPL.format(
        disease_tag=disease_tag, air=SCENARIO, year=year
    )
    if not path.exists():
        print(f"⚠️ Missing disease flow file: {path}")
        return None
    df = pd.read_csv(path, index_col=0)
    df.index   = rename_idx(df.index)
    df.columns = rename_idx(df.columns)
    df = df.loc[~df.index.isin(["total"]), ~df.columns.isin(["total"])]
    return df

def load_local_bydisease(year):
    """
    Reads Step 1's output directly (citysum_bydisease_{air}_{year}.csv),
    which already contains, per (city, disease):
        mo_total_sum             : total PM2.5-attributable patients residing
                                    in this city (place-of-residence burden) —
                                    this IS the "before mobility" demand, since
                                    in the no-mobility counterfactual everyone
                                    seeks care locally.
        cross_patient_total_sum  : of those, the number who travel elsewhere
        local_mo_total           : mo_total_sum - cross_patient_total_sum
                                    = residents who stay and are treated locally
                                    under the ACTUAL/simulated mobility pattern.
    """
    path = CITYSUM_BYDISEASE_DIR / CITYSUM_BYDISEASE_TPL.format(air=SCENARIO, year=year)
    df = pd.read_csv(path)
    df["city"]    = df["city"].str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    df["disease"] = df["disease"].astype(str).str.strip()
    return df

def disease_to_tag(disease):
    return disease.strip().replace(" ", "_").replace("/", "-")

def compute_year_occupancy_days(year):
    """
    Returns a DataFrame indexed by city with columns:
        occ_before_days : sum_d( mo_total_d * LOS_d )
            "Before mobility" = the no-mobility counterfactual, where every
            PM2.5-attributable patient is treated in their city of residence.
        occ_after_days  : sum_d( (local_mo_total_d + inflow_d) * LOS_d )
            "After mobility" = residents who stayed local (local_mo_total_d,
            which already has outflow netted out) PLUS patients who traveled
            IN from other cities (inflow_d). We deliberately use inflow_d
            alone here, NOT (inflow_d - outflow_d): subtracting outflow again
            would double-count it, since local_mo_total_d = mo_total_d -
            outflow_d already removes it once.
    i.e. total PATIENT-DAYS (not yet divided by bed capacity).
    """
    local_df = load_local_bydisease(year)
    diseases = sorted(local_df["disease"].unique().tolist())

    before_acc, after_acc = None, None

    for disease in diseases:
        los = match_los(disease)
        disease_tag = disease_to_tag(disease)

        sub = local_df[local_df["disease"] == disease].groupby("city").sum(numeric_only=True)
        mo_total_d    = sub["mo_total_sum"]
        local_mo_d    = sub["local_mo_total"]

        flow_mat = load_disease_flow_matrix(year, disease_tag)
        if flow_mat is None:
            inflow_d = pd.Series(0.0, index=mo_total_d.index)
        else:
            # column sums = total patients flowing INTO each destination city
            inflow_d = flow_mat.sum(axis=0).groupby(level=0).sum()

        all_cities = mo_total_d.index.union(local_mo_d.index).union(inflow_d.index)
        mo_total_d = mo_total_d.reindex(all_cities, fill_value=0.0)
        local_mo_d = local_mo_d.reindex(all_cities, fill_value=0.0)
        inflow_d   = inflow_d.reindex(all_cities, fill_value=0.0)

        demand_after_d = (local_mo_d + inflow_d).clip(lower=0.0)

        before_days_d = mo_total_d * los
        after_days_d  = demand_after_d * los

        before_acc = before_days_d if before_acc is None else before_acc.add(before_days_d, fill_value=0.0)
        after_acc  = after_days_d  if after_acc  is None else after_acc.add(after_days_d,  fill_value=0.0)

    return pd.DataFrame({"occ_before_days": before_acc, "occ_after_days": after_acc})

# ── 4. Resource & income-class loaders (unchanged) ────────────────────────────
def load_resource():
    df = pd.read_excel(INPUT_FILE, sheet_name="fig4")
    df = df[["city", "beds", "clinics"]].copy()
    df["beds"]    = pd.to_numeric(df["beds"],    errors="coerce")
    df["clinics"] = pd.to_numeric(df["clinics"], errors="coerce")
    df = df.dropna(subset=["city", "beds", "clinics"])
    df = df[(df["beds"] > 0) & (df["clinics"] > 0)]
    df["city"] = df["city"].str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    return df.groupby("city")["beds"].sum()

def load_gdp_class():
    df = pd.read_csv(INCOME_DIR)
    df[INCOME_CITY_COL] = df[INCOME_CITY_COL].str.strip().map(lambda x: CITY_NAME_MAP.get(x, x))
    df = df.dropna(subset=[INCOME_CITY_COL, INCOME_CLASS_COL])
    df["gdp_class"] = pd.Categorical(df[INCOME_CLASS_COL], categories=GDP_CLASS_ORDER, ordered=True)
    return df.set_index(INCOME_CITY_COL)["gdp_class"]

# ── 5. Compute data ────────────────────────────────────────────────────────────
resource   = load_resource()
gdp_class  = load_gdp_class()

frames        = [compute_year(y) for y in YEARS]           # row-1: unchanged
occ_frames    = [compute_year_occupancy_days(y) for y in YEARS]  # row-2: NEW

avg      = pd.concat(frames).groupby(level=0).mean()
occ_days = pd.concat(occ_frames).groupby(level=0).mean()   # avg patient-days across years

common = avg.index.intersection(resource.index).intersection(occ_days.index)
avg      = avg.loc[common]
res      = resource.loc[common]
occ_days = occ_days.loc[common]

# --- NEW: occupancy rate = patient-days / (beds * 365) ---
avg["Occ_before"] = occ_days["occ_before_days"] / (res * 365) * 100   # expressed as %
avg["Occ_after"]  = occ_days["occ_after_days"]  / (res * 365) * 100   # expressed as %
avg["delta_Occ"]  = avg["Occ_after"] - avg["Occ_before"]              # percentage points

avg["region"]    = avg.index.map(region_map)
avg["gdp_class"] = avg.index.map(gdp_class)
avg["beds"]      = res

# ── Export results to CSV ─────────────────────────────────────────────────────
CSV_DIR = OUTFILE.parent
CSV_DIR.mkdir(parents=True, exist_ok=True)

patient_csv_cols = ["mo_total", "local_patient", "inflow", "net", "demand", "beds", "region", "gdp_class"]
patient_csv_cols = [c for c in patient_csv_cols if c in avg.columns]  # guard in case a column is absent
avg[patient_csv_cols].reset_index().rename(columns={"index": "city"}).to_csv(
    CSV_DIR / "Fig4_patient_counts.csv", index=False, float_format="%.4g"
)
print(f"✓ Saved → {CSV_DIR / 'Fig4_patient_counts.csv'}")

rate_csv_cols = ["Occ_before", "Occ_after", "delta_Occ", "beds", "region", "gdp_class"]
avg[rate_csv_cols].reset_index().rename(columns={"index": "city"}).to_csv(
    CSV_DIR / "Fig4_occupancy_rate.csv", index=False, float_format="%.4g"
)
print(f"✓ Saved → {CSV_DIR / 'Fig4_occupancy_rate.csv'}")

# ── NEW: aggregate Occ_before/Occ_after by (income quintile x region) for
#         the before/after dot-plot (panel h). Mean is taken across all
#         cities within each (gdp_class, region) cell.
def build_occupancy_income_region_stats():
    records = avg[["Occ_before", "Occ_after", "region", "gdp_class"]].dropna(subset=["gdp_class"])
    before = (records.groupby(["gdp_class", "region"])["Occ_before"].mean()
                       .unstack(fill_value=np.nan)
                       .reindex(index=GDP_CLASS_ORDER, columns=["East", "West"]))
    after  = (records.groupby(["gdp_class", "region"])["Occ_after"].mean()
                       .unstack(fill_value=np.nan)
                       .reindex(index=GDP_CLASS_ORDER, columns=["East", "West"]))
    before["Overall"] = records.groupby("gdp_class")["Occ_before"].mean().reindex(GDP_CLASS_ORDER)
    after["Overall"]  = records.groupby("gdp_class")["Occ_after"].mean().reindex(GDP_CLASS_ORDER)
    return before, after

occ_before_stats, occ_after_stats = build_occupancy_income_region_stats()

def compute_network_metrics(series):
    clean_s = series.dropna()
    arr = clean_s.values.astype(np.float64)
    arr = arr[arr > 0]
    if len(arr) == 0:
        return 0.0, 0.0
    p = arr / np.sum(arr)
    entropy = -np.sum(p * np.log(p))
    hhi = np.sum(p ** 2)
    return entropy, hhi

entropy_before, hhi_before = compute_network_metrics(avg["Occ_before"])
entropy_after, hhi_after   = compute_network_metrics(avg["Occ_after"])

# ── 5b. Tail-risk / overload analysis ─────────────────────────────────────────
# τ is now expressed on the OCCUPANCY scale. A common clinical/operational
# rule of thumb treats occupancy > ~0.85 (85%) as an overload signal, and
# > 1.0 (100%) as a theoretical capacity breach. We additionally report a
# data-driven percentile threshold for comparability with the original script.
TAU_PERCENTILE = 0.90
tau_percentile_val = avg["Occ_before"].quantile(TAU_PERCENTILE)
TAU_FIXED = 85.0   # fixed clinical reference threshold (85% occupancy; Occ_* now expressed in %)

def overload_stats(series, tau, beds=None):
    excess = (series - tau).clip(lower=0)
    n_over   = int((series > tau).sum())
    pct_over = 100 * n_over / series.notna().sum()
    total_excess = excess.sum() if beds is None else (excess * beds).sum()
    return {
        "n_over_tau":      n_over,
        "pct_cities_over": pct_over,
        "total_excess":    total_excess,
        "p90":             series.quantile(0.90),
        "p95":             series.quantile(0.95),
        "p99":             series.quantile(0.99),
        "max":             series.max(),
    }

print(f"\n=== Occupancy-based tail-risk (data-driven τ = {TAU_PERCENTILE:.0%} of before = {tau_percentile_val:.1f}%) ===")
stats_before = overload_stats(avg["Occ_before"], tau_percentile_val, beds=res)
stats_after  = overload_stats(avg["Occ_after"],  tau_percentile_val, beds=res)
print(f"{'metric':<20}{'before':>12}{'after':>12}{'change':>12}")
for k in stats_before:
    b, a = stats_before[k], stats_after[k]
    print(f"{k:<20}{b:>12.3f}{a:>12.3f}{(a-b):>12.3f}")

print(f"\n=== Occupancy-based tail-risk (fixed clinical τ = {TAU_FIXED:.0f}%) ===")
stats_before_fixed = overload_stats(avg["Occ_before"], TAU_FIXED, beds=res)
stats_after_fixed  = overload_stats(avg["Occ_after"],  TAU_FIXED, beds=res)
for k in stats_before_fixed:
    b, a = stats_before_fixed[k], stats_after_fixed[k]
    print(f"{k:<20}{b:>12.3f}{a:>12.3f}{(a-b):>12.3f}")

relieved   = avg[(avg["Occ_before"] > TAU_FIXED) & (avg["Occ_after"] <= TAU_FIXED)]
newly_over = avg[(avg["Occ_before"] <= TAU_FIXED) & (avg["Occ_after"] > TAU_FIXED)]
print(f"\nCities relieved from >85% occupancy: {len(relieved)}")
print(f"Cities newly pushed above 85% occupancy: {len(newly_over)}")

# ── 6. Merge with shapefile ────────────────────────────────────────────────────
avg_reset = avg[["mo_total", "demand", "net", "Occ_before", "Occ_after",
                 "delta_Occ", "region", "gdp_class"]].reset_index().rename(columns={"index": "city"})
shp = city_shp.merge(avg_reset, left_on="English", right_on="city", how="left")

# ── 6b. Color-scale ranges ─────────────────────────────────────────────────────
vmin_occ = 0
vmax_occ = max(avg["Occ_before"].quantile(0.98), avg["Occ_after"].quantile(0.98))

vmin_patient = 0
vmax_patient = max(avg["mo_total"].quantile(0.98), avg["demand"].quantile(0.98))

dmax_net = shp["net"].abs().quantile(0.99)
dmax_occ = shp["delta_Occ"].abs().quantile(0.99)

# ── 7. Nanhai inset helper ─────────────────────────────────────────────────────
def _add_nanhai_inset(parent_ax, target_shp, col, norm, cmap):
    x1, y1, x2, y2 = _NANHAI_BOUNDS
    axins = parent_ax.inset_axes([0.1, 0.01, 0.18, 0.22])
    axins.set_facecolor("white")
    target_shp.plot(column=col, ax=axins, cmap=cmap, norm=norm,
                    linewidth=0, missing_kwds={"color": C_NODATA, "edgecolor": "none"})
    china_border.plot(ax=axins, facecolor="none", edgecolor="black", linewidth=0.2)
    jiudanline.plot(ax=axins, edgecolor="black", linewidth=0.4)
    axins.set_xlim(x1, x2)
    axins.set_ylim(y1, y2)
    axins.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in axins.spines.values():
        spine.set_linewidth(0.5)
        spine.set_color("black")

# ── 8. Map drawing helpers ─────────────────────────────────────────────────────
def _map_base(ax, col, norm, cmap, title, panel, show_hu_label=False):
    shp.plot(column=col, ax=ax, cmap=cmap, norm=norm,
             linewidth=0.15, edgecolor="#AAAAAA",
             missing_kwds={"color": C_NODATA, "edgecolor": "#AAAAAA"})
    china_border.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.15)
    jiudanline.plot(ax=ax, edgecolor="black", linewidth=0.25)
    ax.plot(_HHY_X, _HHY_Y, color="black", linewidth=0.7,
            linestyle="--", dashes=(4, 3), zorder=5)
    xmin, ymin, xmax, ymax = china_border.total_bounds
    height = ymax - ymin
    width  = xmax - xmin
    ax.set_xlim(xmin, xmax + width * 0.04)
    ax.set_ylim(ymin + height * 0.20, ymax + height * 0.01)
    ax.set_axis_off()
    ax.text(0.5, 1.01, title,
            transform=ax.transAxes, fontsize=7, fontweight="bold", va="bottom", ha="center")
    _add_nanhai_inset(ax, shp, col, norm, cmap)

def draw_seq_map(ax, col, title, panel, norm, cmap="YlOrRd", stat_text=None, show_hu_label=False):
    _map_base(ax, col, norm, cmap, title, panel, show_hu_label)
    if stat_text:
        ax.text(0.05, 0.04, stat_text, transform=ax.transAxes,
                fontsize=5.5, color="#333333", va="bottom", ha="left")

def draw_diff_map(ax, col, title, panel, dmax, cmap="RdBu_r", show_hu_label=False):
    norm = TwoSlopeNorm(vmin=-dmax, vcenter=0, vmax=dmax)
    _map_base(ax, col, norm, cmap, title, panel, show_hu_label)
    return norm, cmap

# ── 9. Grid layout: 3 rows (maps, maps, income-group bars) ────────────────────
fig = plt.figure(figsize=(18 / 2.54, 20 / 2.54), dpi=400, facecolor="white")

gs_row1 = GridSpec(1, 3, figure=fig, wspace=0.10, left=0, right=0.97, top=1, bottom=0.67)
gs_row2 = GridSpec(1, 3, figure=fig, wspace=0.10, left=0, right=0.97, top=0.63, bottom=0.33)
from matplotlib.gridspec import GridSpecFromSubplotSpec

gs_row3 = GridSpec(1, 3, figure=fig, wspace=0.12, left=0.09, right=0.98, top=0.24, bottom=0.06)

ax_r1a = fig.add_subplot(gs_row1[0, 0])
ax_r1b = fig.add_subplot(gs_row1[0, 1])
ax_r1c = fig.add_subplot(gs_row1[0, 2])

ax_r2a = fig.add_subplot(gs_row2[0, 0])
ax_r2b = fig.add_subplot(gs_row2[0, 1])
ax_r2c = fig.add_subplot(gs_row2[0, 2])

ax_h_east, ax_h_west, ax_h_overall = [fig.add_subplot(gs_row3[0, j]) for j in range(3)]

# ── 10. Row 1 — patient counts (unchanged) ────────────────────────────────────
norm_patient = Normalize(vmin=vmin_patient, vmax=vmax_patient)
draw_seq_map(ax_r1a, "mo_total", "Patients before mobility", "a", norm_patient,
             cmap="YlOrRd", show_hu_label=True)
draw_seq_map(ax_r1b, "demand", "Patients after mobility", "b", norm_patient, cmap="YlOrRd")
norm_net, cmap_net = draw_diff_map(ax_r1c, "net", "Net inflow patients", "c", dmax_net)

ROW_LABEL_X = 0.005   # shared x position so a/b/c line up in one column
fig.text(ROW_LABEL_X, 0.99, "a", fontsize=10, fontweight="bold", va="top", ha="left")

# ── 11. Row 2 — BED OCCUPANCY RATE (NEW) ──────────────────────────────────────
norm_occ = Normalize(vmin=vmin_occ, vmax=vmax_occ)
draw_seq_map(ax_r2a, "Occ_before", "Bed occupancy rate before mobility", "d", norm_occ, cmap="YlOrRd")
draw_seq_map(ax_r2b, "Occ_after",  "Bed occupancy rate after mobility",  "e", norm_occ, cmap="YlOrRd")
norm_occ_diff, cmap_occ_diff = draw_diff_map(ax_r2c, "delta_Occ", "Change in bed occupancy rate", "f", dmax_occ)

fig.text(ROW_LABEL_X, 0.64, "b", fontsize=10, fontweight="bold", va="top", ha="left")


# ── 12. Colorbars for row 1 and row 2 ─────────────────────────────────────────
fig.canvas.draw()

def add_row_colorbars(ax_left, ax_mid, ax_right, norm_seq, cmap_seq, norm_diff, cmap_diff,
                       label_seq, label_diff, y_offset=0.02):
    pos_l = ax_left.get_position()
    pos_m = ax_mid.get_position()
    pos_r = ax_right.get_position()
    cbar_h = 0.010
    cbar_bottom = pos_l.y0 - y_offset

    cbar_ax_lm = fig.add_axes([pos_l.x0 + 0.05, cbar_bottom,
                               (pos_m.x1 * 0.8 - (pos_l.x0 + 0.05)) * 0.9, cbar_h])
    sm_lm = mcm.ScalarMappable(cmap=cmap_seq, norm=norm_seq)
    sm_lm.set_array([])
    cbar_lm = fig.colorbar(sm_lm, cax=cbar_ax_lm, orientation="horizontal")
    cbar_lm.ax.tick_params(labelsize=5.5)
    cbar_lm.set_label(label_seq, fontsize=5.5, labelpad=2)

    rect_ax = fig.add_axes([(pos_m.x1 * 0.8 - (pos_l.x0 + 0.05)) * 1.15, cbar_bottom, 0.025, cbar_h])
    rect_ax.patch.set_facecolor(C_NODATA)
    rect_ax.patch.set_edgecolor("#AAAAAA")
    rect_ax.patch.set_linewidth(0.3)
    rect_ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    fig.text(pos_m.x1 * 0.88, cbar_bottom + 0.002, "No hospital data", fontsize=5.5, color="#333333", va="bottom")

    cbar_ax_r = fig.add_axes([pos_r.x0 * 1.15, cbar_bottom, (pos_r.x1 * 0.95 - pos_r.x0 * 1.05) * 0.75, cbar_h])
    sm_r = mcm.ScalarMappable(cmap=cmap_diff, norm=norm_diff)
    sm_r.set_array([])
    cbar_r = fig.colorbar(sm_r, cax=cbar_ax_r, orientation="horizontal")
    cbar_r.ax.tick_params(labelsize=5.5)
    cbar_r.set_label(label_diff, fontsize=5.5, labelpad=2)

add_row_colorbars(ax_r1a, ax_r1b, ax_r1c, norm_patient, "YlOrRd", norm_net, cmap_net,
                   "Patients", "Patients", y_offset=0.025)
add_row_colorbars(ax_r2a, ax_r2b, ax_r2c, norm_occ, "YlOrRd", norm_occ_diff, cmap_occ_diff,
                   "Bed occupancy rate (pp)", "Bed occupancy rate (pp)", y_offset=0.025)

# ── 13. Bar style helper ───────────────────────────────────────────────────────
def style_bar(ax, title, ylabel):
    ax.text(-0.12, 1.05, title, transform=ax.transAxes, fontsize=8, fontweight="bold", va="bottom", ha="left")
    ax.set_ylabel(ylabel, fontsize=6.5)
    ax.axhline(0, color="#888", lw=0.8, ls="--", zorder=1)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="x", length=0, labelsize=6.5)
    ax.tick_params(axis="y", labelsize=6)
    ax.grid(axis="y", color="#E0E0E0", lw=0.5, zorder=0)
    ax.set_facecolor("white")

# ── 14. Row 3 — h: BEFORE/AFTER occupancy dot-plot, split East/West/Overall ───
STATE_COLORS = {"before": "#4e79a7", "after": "#e15759"}  # same palette as your Fig2 YEAR_COLORS

def draw_occupancy_dots(axes_row, before_df, after_df, ylabel):
    """
    Connected dot plot: one 'before mobility' dot (blue) and one
    'after mobility' dot (red) per income quintile, joined by a dashed grey
    line; % change annotated in red (increase) or black (decrease).
    Mirrors the visual style of your Fig2 draw_income_dots function.

    axes_row   : [ax_East, ax_West, ax_Overall]
    before_df  : DataFrame(index=GDP_CLASS_ORDER, columns=["East","West","Overall"])
    after_df   : same shape, "after mobility" values
    """
    x = np.arange(len(GDP_CLASS_ORDER))
    cols = ["East", "West", "Overall"]

    all_vals = np.concatenate([
        before_df[cols].values.flatten(), after_df[cols].values.flatten()
    ])
    all_vals = all_vals[~np.isnan(all_vals)]
    ymax_val = np.nanmax(all_vals) * 1.35 if len(all_vals) else 1

    for j, (ax, col) in enumerate(zip(axes_row, cols)):
        v0 = before_df[col].reindex(GDP_CLASS_ORDER).values
        v1 = after_df[col].reindex(GDP_CLASS_ORDER).values

        for xi, (val0, val1) in enumerate(zip(v0, v1)):
            if np.isnan(val0) or np.isnan(val1):
                continue
            ax.plot([xi, xi], [val0, val1], color="#aaaaaa", linewidth=1.2,
                    linestyle=(0, (3, 2)), zorder=1)
            ax.scatter(xi, val0, s=14, color=STATE_COLORS["before"], zorder=3, linewidths=0)
            ax.scatter(xi, val1, s=14, color=STATE_COLORS["after"],  zorder=3, linewidths=0)

            if val0 > 0:
                pct = (val1 - val0) / val0 * 100
                label = f"{pct:+.1f}%"
                color = "#C0392B" if pct > 0 else "#2c2c2c"
                ymid = (val0 + val1) / 2
                ax.text(xi + 0.15, ymid, label, fontsize=5.5, color=color,
                        va="center", ha="left", zorder=4)

        ax.set_xticks(x)
        ax.set_xticklabels([GDP_CLASS_SHORT[g] for g in GDP_CLASS_ORDER], fontsize=6, rotation=20, ha="right")
        ax.set_ylim(0, ymax_val)
        ax.tick_params(axis="y", labelsize=6)
        ax.spines[["top", "right"]].set_visible(False)
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))
        ax.grid(axis="y", color="#E0E0E0", lw=0.5, zorder=0)
        ax.set_facecolor("white")
        ax.text(0.04, 0.97, col, transform=ax.transAxes, fontsize=6.5,
                fontstyle="italic", va="top", ha="left", color="#555555")

        if j == 0:
            ax.set_ylabel(ylabel, fontsize=6.5)
        else:
            ax.set_yticklabels([])

    pass  # row-level letter "c" and the shared title are drawn once, at figure level, below
    axes_row[-1].legend(
        handles=[
            plt.Line2D([0], [0], marker="o", color="w",
                       markerfacecolor=STATE_COLORS["before"], markersize=4, label="Before mobility"),
            plt.Line2D([0], [0], marker="o", color="w",
                       markerfacecolor=STATE_COLORS["after"], markersize=4, label="After mobility"),
        ],
        fontsize=5.5, frameon=False, loc="upper right"
    )

draw_occupancy_dots([ax_h_east, ax_h_west, ax_h_overall],
                     occ_before_stats, occ_after_stats, "Bed occupancy rate (pp)")

fig.text(ROW_LABEL_X, 0.25, "c", fontsize=10, fontweight="bold", va="top", ha="left")



# ── 16. Export and render ─────────────────────────────────────────────────────
OUTFILE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTFILE, dpi=400, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved → {OUTFILE}")


✓ Saved → /Users/shirley/Desktop/plots_V2/Fig4_patient_counts.csv
✓ Saved → /Users/shirley/Desktop/plots_V2/Fig4_occupancy_rate.csv

=== Occupancy-based tail-risk (data-driven τ = 90% of before = 4.9%) ===
metric                    before       after      change
n_over_tau                35.000      22.000     -13.000
pct_cities_over           10.000       6.286      -3.714
total_excess          204719.120  135210.989  -69508.131
p90                        4.912       4.305      -0.606
p95                        5.950       5.371      -0.579
p99                       10.537       8.192      -2.345
max                       22.008      18.631      -3.377

=== Occupancy-based tail-risk (fixed clinical τ = 85%) ===
n_over_tau                 0.000       0.000       0.000
pct_cities_over            0.000       0.000       0.000
total_excess               0.000       0.000       0.000
p90                        4.912       4.305      -0.606
p95                        5.950       5.371      

/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_38455/1838995062.py:394: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  before = (records.groupby(["gdp_class", "region"])["Occ_before"].mean()
/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_38455/1838995062.py:397: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  after  = (records.groupby(["gdp_class", "region"])["Occ_after"].mean()
/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_38455/1838995062.py:400: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observ